# Exercises XP: Machine Learning Fundamentals



In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           roc_auc_score, average_precision_score, confusion_matrix, 
                           classification_report, silhouette_score)
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")


Libraries imported successfully


## Exercise 1: Defining the Problem and Data Collection for Loan Default Prediction



### Problem Statement and Data Collection Plan

**Problem Statement:**
The objective is to predict the probability of loan default (binary classification) to support financial institutions in making informed lending decisions and minimizing credit risk exposure.

**Data Types Required:**

**1. Demographic Information:**
Age, employment length, and education level provide insights into borrower stability and earning potential, which directly correlate with repayment ability.

**2. Financial Information:**
Annual income, debt-to-income ratio, and existing credit obligations indicate the borrower's financial capacity to handle additional debt payments.

**3. Credit History:**
Credit score, number of delinquencies, and payment history reflect past financial behavior and creditworthiness patterns.

**4. Loan Characteristics:**
Loan amount, interest rate, loan term, and purpose influence the difficulty of repayment and risk profile of the loan.

**5. Asset Information:**
Home ownership status and collateral value provide security measures and indicate financial stability.

**Data Sources:**

**Internal Financial Institution Records:** Historical loan performance data, customer account information, and transaction histories can be accessed through the institution's database systems.

**Credit Bureau Reports:** Comprehensive credit scores, payment histories, and credit utilization data can be obtained through partnerships with major credit reporting agencies like Experian, Equifax, and TransUnion.

**Government Databases:** Employment verification and income validation can be sourced through tax records and employment databases with appropriate authorization.

**Third-Party Data Providers:** Additional demographic and behavioral data can be integrated through specialized financial data vendors and alternative credit scoring companies.

**Risks and Constraints:**
Privacy regulations such as GDPR and CCPA require explicit consent and data anonymization protocols. Regulatory compliance with fair lending practices mandates bias testing and model transparency. Data quality issues including missing values, outdated information, and reporting inconsistencies can significantly impact model performance. Sampling bias may occur if historical data overrepresents certain demographic groups or time periods. Data governance frameworks must ensure proper data lineage, access controls, and model audit trails throughout the development and deployment lifecycle.

## Exercise 2: Feature Selection and Model Choice for Loan Default Prediction

**Instructions:**
- Identify which features might be most relevant for predicting loan defaults
- Justify your choice of features
- Explain feature encoding and missing value handling strategies

In [ ]:
example_columns = [
    "age", "employment_length", "annual_income", "credit_score", "loan_amount", "interest_rate",
    "debt_to_income", "num_delinquencies", "num_open_accounts", "total_utilization", 
    "home_ownership", "purpose", "term", "application_type", "state", "zip_code"
]

# Create placeholder DataFrame
df = pd.DataFrame(columns=example_columns)

# Selected features based on relevance to default prediction
selected_features = [
    "credit_score", "debt_to_income", "annual_income", "loan_amount", "interest_rate",
    "employment_length", "num_delinquencies", "total_utilization", "home_ownership",
    "age", "purpose", "term"
]

print("Dataset structure created with", len(example_columns), "potential features")
print("Selected", len(selected_features), "most relevant features for loan default prediction")
print("\nSelected features:")
for feature in selected_features:
    print(f"- {feature}")

Dataset structure created with 16 potential features
Selected 12 most relevant features for loan default prediction

Selected features:
- credit_score
- debt_to_income
- annual_income
- loan_amount
- interest_rate
- employment_length
- num_delinquencies
- total_utilization
- home_ownership
- age
- purpose
- term


## Exercise 3: Model Selection and Evaluation for Loan Default Prediction

**Instructions:** Build and evaluate different machine learning models for loan default prediction. Compare their performance using appropriate metrics and select the best model for deployment.

**Requirements:**
1. Implement at least 3 different classification algorithms
2. Use appropriate evaluation metrics for imbalanced classification
3. Perform hyperparameter tuning on the best performing model
4. Provide business interpretation of model results

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# Simulate dataset 
np.random.seed(42)
n_samples = 10000

# Generate features 
X = np.random.rand(n_samples, 12)
# Create realistic target with class imbalance 
y = np.random.binomial(1, 0.1, n_samples)

# Feature names for our selected features
feature_names = [
    'credit_score', 'debt_to_income', 'annual_income', 'loan_amount',
    'interest_rate', 'employment_length', 'delinquencies', 'total_utilization',
    'home_ownership', 'age', 'loan_purpose', 'term_length'
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"Default rate in training set: {y_train.mean():.3f}")
print(f"Default rate in test set: {y_test.mean():.3f}")

models = {
    'Logistic Regression': LogisticRegression(random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

model_results = {}

for name, model in models.items():
    print(f"\n--- {name} ---")
    
    #  scaled data for logistic regression, original for tree-based models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    #  metrics
    auc_score = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    model_results[name] = {
        'model': model,
        'auc_score': auc_score,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"AUC Score: {auc_score:.3f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Compare model performance
print("\n=== Model Comparison ===")
for name, results in model_results.items():
    print(f"{name}: AUC = {results['auc_score']:.3f}")

#  best model
best_model_name = max(model_results.keys(), key=lambda x: model_results[x]['auc_score'])
print(f"\nBest performing model: {best_model_name}")

Training set size: 8000 samples
Test set size: 2000 samples
Default rate in training set: 0.095
Default rate in test set: 0.095

--- Logistic Regression ---
AUC Score: 0.470

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.52      0.66      1810
           1       0.09      0.44      0.15       190

    accuracy                           0.51      2000
   macro avg       0.49      0.48      0.40      2000
weighted avg       0.82      0.51      0.61      2000


--- Random Forest ---
AUC Score: 0.506

Classification Report:
              precision    recall  f1-score   support

           0       0.91      1.00      0.95      1810
           1       0.00      0.00      0.00       190

    accuracy                           0.91      2000
   macro avg       0.45      0.50      0.48      2000
weighted avg       0.82      0.91      0.86      2000


--- Gradient Boosting ---
AUC Score: 0.479

Classification Report:
              pr

In [4]:
# Hyperparameter tuning 
print(f"\n=== Hyperparameter Tuning for {best_model_name} ===")

if best_model_name == 'Random Forest':
    param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    base_model = RandomForestClassifier(random_state=42, class_weight='balanced')
    X_tune = X_train
    
elif best_model_name == 'Gradient Boosting':
    param_grid = {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }
    base_model = GradientBoostingClassifier(random_state=42)
    X_tune = X_train
    
else:  # Logistic Regression
    param_grid = {
        'C': [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    }
    base_model = LogisticRegression(random_state=42, class_weight='balanced')
    X_tune = X_train_scaled

#  grid search with cross-validation
grid_search = GridSearchCV(
    base_model, 
    param_grid, 
    cv=5, 
    scoring='roc_auc', 
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_tune, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation AUC: {grid_search.best_score_:.3f}")

# Evaluate tuned model on test set
best_model = grid_search.best_estimator_

if best_model_name == 'Logistic Regression':
    y_pred_tuned = best_model.predict(X_test_scaled)
    y_pred_proba_tuned = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    y_pred_tuned = best_model.predict(X_test)
    y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

tuned_auc = roc_auc_score(y_test, y_pred_proba_tuned)
print(f"\nTuned model test AUC: {tuned_auc:.3f}")
print(f"Improvement over baseline: {tuned_auc - model_results[best_model_name]['auc_score']:.3f}")

print("\n=== Final Model Performance ===")
print(classification_report(y_test, y_pred_tuned))

print("\n=== Business Interpretation ===")
print(f"The final {best_model_name} model achieves an AUC of {tuned_auc:.3f}.")
print("This means the model can distinguish between defaulters and non-defaulters")
print(f"with {tuned_auc*100:.1f}% accuracy in ranking risk.")
print("\nBusiness Impact:")
print("- Higher AUC enables better risk assessment and loan pricing")
print("- Model can identify high-risk applications for manual review")
print("- Potential to reduce default rates while maintaining loan volume")

NameError: name 'best_model_name' is not defined

## Exercise 4: Machine Learning Paradigm Selection

**Instructions:** For each scenario below, identify the most appropriate machine learning paradigm (Supervised, Unsupervised, or Reinforcement Learning) and justify your choice. Explain what type of problem it represents and what approach you would take.

**Scenarios:**
1. **Stock Price Prediction:** Predicting tomorrow's closing price of Apple stock
2. **Library Book Organization:** Grouping books in a library based on reading patterns
3. **Robot Navigation:** Teaching a robot to navigate through a maze to reach a goal

**Requirements:**
- Identify the ML paradigm for each scenario
- Explain the reasoning behind your choice
- Describe the input data, output, and learning approach
- Suggest specific algorithms that would be appropriate

In [5]:

print("=== MACHINE LEARNING PARADIGM SELECTION ===\n")

print("1. STOCK PRICE PREDICTION")
print("Paradigm: SUPERVISED LEARNING")
print("Reasoning:")
print("- We have historical stock prices as labeled training data")
print("- The goal is to predict a continuous numerical value (stock price)")
print("- This is a regression problem with input-output pairs")
print("\nProblem Type: Time Series Regression")
print("Input Data: Historical stock prices, trading volumes, technical indicators, market sentiment")
print("Output: Predicted closing price for tomorrow")
print("Approach: Train on historical data to learn patterns between features and future prices")
print("Suitable Algorithms:")
print("  • Linear Regression")
print("  • Random Forest Regressor")
print("  • LSTM (Long Short-Term Memory) Neural Networks")
print("  • ARIMA (AutoRegressive Integrated Moving Average)")
print("  • Support Vector Regression")

print("\n" + "="*60 + "\n")

#  Library Book Organization
print("2. LIBRARY BOOK ORGANIZATION")
print("Paradigm: UNSUPERVISED LEARNING")
print("Reasoning:")
print("- No predefined categories or labels for book groupings")
print("- Goal is to discover hidden patterns in reading behavior")
print("- We want to find natural clusters based on similarities")
print("\nProblem Type: Clustering Analysis")
print("Input Data: Reading patterns, borrowing frequency, book metadata, user demographics")
print("Output: Groups/clusters of books with similar reading patterns")
print("Approach: Analyze data to find natural groupings without predefined labels")
print("Suitable Algorithms:")
print("  • K-Means Clustering")
print("  • Hierarchical Clustering")
print("  • DBSCAN (Density-Based Clustering)")
print("  • Gaussian Mixture Models")
print("  • Principal Component Analysis (for dimensionality reduction)")

print("\n" + "="*60 + "\n")

#  Robot Navigation
print("3. ROBOT NAVIGATION IN MAZE")
print("Paradigm: REINFORCEMENT LEARNING")
print("Reasoning:")
print("- Robot learns through trial and error interaction with environment")
print("- No labeled training data available initially")
print("- Goal-oriented learning with rewards and penalties")
print("- Sequential decision-making problem")
print("\nProblem Type: Sequential Decision Making")
print("Input Data: Current state (position, sensor readings)")
print("Output: Action to take (move up, down, left, right)")
print("Approach: Agent learns optimal policy through exploration and exploitation")
print("Learning Components:")
print("  • State: Robot's current position and environment information")
print("  • Actions: Possible movements (up, down, left, right)")
print("  • Rewards: Positive for reaching goal, negative for hitting walls")
print("  • Policy: Strategy for choosing actions based on current state")
print("Suitable Algorithms:")
print("  • Q-Learning")
print("  • Deep Q-Networks (DQN)")
print("  • Policy Gradient Methods")
print("  • Actor-Critic Methods")
print("  • Monte Carlo Tree Search")

print("\n" + "="*60)
print("\nSUMMARY:")
print("• Supervised Learning: When you have labeled data and want to predict outcomes")
print("• Unsupervised Learning: When you want to discover patterns without labels")
print("• Reinforcement Learning: When learning through interaction and feedback")

=== MACHINE LEARNING PARADIGM SELECTION ===

1. STOCK PRICE PREDICTION
Paradigm: SUPERVISED LEARNING
Reasoning:
- We have historical stock prices as labeled training data
- The goal is to predict a continuous numerical value (stock price)
- This is a regression problem with input-output pairs

Problem Type: Time Series Regression
Input Data: Historical stock prices, trading volumes, technical indicators, market sentiment
Output: Predicted closing price for tomorrow
Approach: Train on historical data to learn patterns between features and future prices
Suitable Algorithms:
  • Linear Regression
  • Random Forest Regressor
  • LSTM (Long Short-Term Memory) Neural Networks
  • ARIMA (AutoRegressive Integrated Moving Average)
  • Support Vector Regression


2. LIBRARY BOOK ORGANIZATION
Paradigm: UNSUPERVISED LEARNING
Reasoning:
- No predefined categories or labels for book groupings
- Goal is to discover hidden patterns in reading behavior
- We want to find natural clusters based on simila

## Exercise 5: Evaluation Strategies for Different ML Paradigms

**Instructions:** Design appropriate evaluation strategies for different machine learning paradigms. Explain what metrics and approaches you would use for each type of learning.

**Requirements:**
1. **Supervised Learning Evaluation:** Define metrics for both classification and regression problems
2. **Unsupervised Learning Evaluation:** Explain how to evaluate clustering and dimensionality reduction
3. **Reinforcement Learning Evaluation:** Describe evaluation approaches for RL agents
4. **Cross-validation strategies:** When and how to use different validation techniques

In [6]:
# Exercise 5: Evaluation Strategies for Different ML Paradigms

from sklearn.metrics import silhouette_score, adjusted_rand_score, calinski_harabasz_score
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import numpy as np
import matplotlib.pyplot as plt

print("=== EVALUATION STRATEGIES FOR MACHINE LEARNING PARADIGMS ===\n")

# 1. SUPERVISED LEARNING EVALUATION
print("1. SUPERVISED LEARNING EVALUATION")
print("\nA. CLASSIFICATION METRICS:")
print("   • Accuracy: Overall correctness (TP+TN)/(TP+TN+FP+FN)")
print("   • Precision: True positives / (True positives + False positives)")
print("   • Recall (Sensitivity): True positives / (True positives + False negatives)")
print("   • F1-Score: Harmonic mean of precision and recall")
print("   • AUC-ROC: Area under ROC curve (discrimination ability)")
print("   • AUC-PR: Area under Precision-Recall curve (for imbalanced data)")
print("   • Specificity: True negatives / (True negatives + False positives)")
print("   • Cohen's Kappa: Agreement between predicted and actual, adjusted for chance")

print("\nB. REGRESSION METRICS:")
print("   • Mean Absolute Error (MAE): Average absolute difference")
print("   • Mean Squared Error (MSE): Average squared difference")
print("   • Root Mean Squared Error (RMSE): Square root of MSE")
print("   • R-squared (R²): Proportion of variance explained")
print("   • Mean Absolute Percentage Error (MAPE): Percentage-based error")
print("   • Adjusted R²: R² adjusted for number of predictors")

print("\nC. CROSS-VALIDATION STRATEGIES:")
print("   • K-Fold CV: Split data into k folds, train on k-1, test on 1")
print("   • Stratified K-Fold: Maintains class distribution in each fold")
print("   • Time Series Split: Respects temporal order for time-dependent data")
print("   • Leave-One-Out CV: Each sample used once as test set")
print("   • Group K-Fold: Ensures same group doesn't appear in train and test")

print("\n" + "="*70)

# 2. UNSUPERVISED LEARNING EVALUATION
print("\n2. UNSUPERVISED LEARNING EVALUATION")
print("\nA. CLUSTERING EVALUATION:")

# Demonstrate clustering evaluation with synthetic data
X_cluster, y_true = make_blobs(n_samples=300, centers=4, random_state=42)
kmeans = KMeans(n_clusters=4, random_state=42)
y_pred = kmeans.fit_predict(X_cluster)

silhouette_avg = silhouette_score(X_cluster, y_pred)
calinski_harabasz = calinski_harabasz_score(X_cluster, y_pred)
adjusted_rand = adjusted_rand_score(y_true, y_pred)

print(f"\nCLUSTERING METRICS EXAMPLE:")
print(f"   • Silhouette Score: {silhouette_avg:.3f} (range: -1 to 1, higher is better)")
print(f"   • Calinski-Harabasz Index: {calinski_harabasz:.1f} (higher is better)")
print(f"   • Adjusted Rand Index: {adjusted_rand:.3f} (range: -1 to 1, higher is better)")

print("\nINTERNAL METRICS (no ground truth needed):")
print("   • Silhouette Score: Measures how similar objects are to their cluster vs other clusters")
print("   • Calinski-Harabasz Index: Ratio of between-cluster to within-cluster dispersion")
print("   • Davies-Bouldin Index: Average similarity between clusters (lower is better)")
print("   • Inertia/Within-cluster sum of squares: Compactness of clusters")

print("\nEXTERNAL METRICS (require ground truth):")
print("   • Adjusted Rand Index: Similarity between predicted and true clustering")
print("   • Normalized Mutual Information: Information shared between clusterings")
print("   • Homogeneity: Each cluster contains only members of a single class")
print("   • Completeness: All members of a class are assigned to the same cluster")

print("\nB. DIMENSIONALITY REDUCTION EVALUATION:")
print("   • Explained Variance Ratio: Proportion of variance retained")
print("   • Reconstruction Error: Difference between original and reconstructed data")
print("   • Trustworthiness: How well local neighborhoods are preserved")
print("   • Continuity: How well the mapping preserves local neighborhoods")

print("\n" + "="*70)

# 3. REINFORCEMENT LEARNING EVALUATION
print("\n3. REINFORCEMENT LEARNING EVALUATION")
print("\nA. PERFORMANCE METRICS:")
print("   • Cumulative Reward: Total reward over an episode")
print("   • Average Return: Mean cumulative reward over multiple episodes")
print("   • Success Rate: Percentage of episodes where goal is achieved")
print("   • Episode Length: Number of steps to complete task")
print("   • Learning Curve: Performance improvement over training time")
print("   • Sample Efficiency: How quickly agent learns from experience")
print("\nB. EVALUATION APPROACHES:")
print("   • Online Evaluation: Assess performance during training")
print("   • Offline Evaluation: Test trained agent on separate episodes")
print("   • Cross-evaluation: Test on different environments/scenarios")
print("   • Human Evaluation: Compare against human performance")
print("   • Ablation Studies: Remove components to understand contributions")

print("\nC. SPECIFIC RL CONSIDERATIONS:")
print("   • Exploration vs Exploitation: Balance between trying new actions and using known good ones")
print("   • Generalization: Performance on unseen states/environments")
print("   • Robustness: Performance under different conditions")
print("   • Safety: Avoid harmful actions during exploration")

print("\n" + "="*70)

# 4. GENERAL EVALUATION PRINCIPLES
print("\n4. GENERAL EVALUATION PRINCIPLES")
print("\nA. DATA SPLITTING:")
print("   • Training Set (60-70%): Used to train the model")
print("   • Validation Set (15-20%): Used for hyperparameter tuning")
print("   • Test Set (15-20%): Used for final unbiased evaluation")

print("\nB. EVALUATION BEST PRACTICES:")
print("   • Never use test data for model selection or tuning")
print("   • Ensure data leakage doesn't occur between sets")
print("   • Use stratification for imbalanced datasets")
print("   • Consider temporal order for time series data")
print("   • Report confidence intervals for metrics")
print("   • Use multiple random seeds for robust results")

print("\nC. CHOOSING APPROPRIATE METRICS:")
print("   • Business Context: Align metrics with business objectives")
print("   • Data Characteristics: Consider class imbalance, noise, etc.")
print("   • Problem Type: Classification, regression, clustering, etc.")
print("   • Stakeholder Understanding: Use interpretable metrics when possible")

print("\nD. COMMON PITFALLS:")
print("   • Data Snooping: Looking at test data multiple times")
print("   • Overfitting to Validation Set: Excessive hyperparameter tuning")
print("   • Ignoring Class Imbalance: Using accuracy for imbalanced datasets")
print("   • Temporal Leakage: Using future information to predict past")
print("   • Selection Bias: Non-representative train/test splits")

print("\n" + "="*70)
print("\nCONCLUSION:")
print("Proper evaluation is crucial for building reliable ML systems.")
print("Choose metrics that align with your problem and business objectives.")
print("Always validate on unseen data and consider real-world deployment challenges.")

=== EVALUATION STRATEGIES FOR MACHINE LEARNING PARADIGMS ===

1. SUPERVISED LEARNING EVALUATION

A. CLASSIFICATION METRICS:
   • Accuracy: Overall correctness (TP+TN)/(TP+TN+FP+FN)
   • Precision: True positives / (True positives + False positives)
   • Recall (Sensitivity): True positives / (True positives + False negatives)
   • F1-Score: Harmonic mean of precision and recall
   • AUC-ROC: Area under ROC curve (discrimination ability)
   • AUC-PR: Area under Precision-Recall curve (for imbalanced data)
   • Specificity: True negatives / (True negatives + False positives)
   • Cohen's Kappa: Agreement between predicted and actual, adjusted for chance

B. REGRESSION METRICS:
   • Mean Absolute Error (MAE): Average absolute difference
   • Mean Squared Error (MSE): Average squared difference
   • Root Mean Squared Error (RMSE): Square root of MSE
   • R-squared (R²): Proportion of variance explained
   • Mean Absolute Percentage Error (MAPE): Percentage-based error
   • Adjusted R²: R² 